# 06 – Knowledge Transfer: Streamlit-App

**Projekt:** WealthScope AI 1.0
**Methode:** QUA³CK · reproduzierbarer Out-of-Time-Benchmark
**Hinweis:** Wissenschaftlicher Prototyp, keine Anlageberatung.

## Von Analyse zu Wissen

Notebooks dokumentieren die Herleitung; die App macht sie interaktiv. Der
Knowledge-Schritt trennt Prognose, Erklärung und Handlungshilfe sichtbar.

## Lernziele

Nach diesem Notebook könnt ihr:

- erklären, warum ein negatives Ergebnis besonders sorgfältig kommuniziert werden muss
- die Trennung von Rechnen, Visualisieren und Sprache-Erzeugen benennen
- prüfen, ob eine App ihre eigenen Grenzen sichtbar macht

### Die schwierigste Kommunikationsaufgabe

Ein Prototyp, der ein starkes Signal zeigt, verkauft sich von selbst. Dieser
zeigt keines — und muss trotzdem verständlich machen, warum das die richtige
Antwort ist. Daraus folgen drei Gestaltungsentscheidungen der App:

1. **Die Baseline steht neben jeder Kennzahl.** „51,9 %" allein wäre irreführend,
   „51,9 % gegen eine Baseline von 59,1 %" ist eine Aussage.
2. **Kein Wert ist im Code hartkodiert.** Alle Seiten lesen `models/*.json`.
   Ändert sich das Modell, ändert sich die App — ein Widerspruch zwischen
   Dokumentation und Anzeige kann nicht entstehen.
3. **Der Assistent ist vom Prognosemodell getrennt.** Er erklärt den
   berechneten Kontext, er erzeugt keine Vorhersage. Diese Trennung ist die
   Voraussetzung dafür, dass „KI" im Projekt kein Sammelbegriff bleibt.

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "wealthscope_features.parquet"
DIAGNOSTICS_PATH = PROJECT_ROOT / "models" / "diagnostics.json"
EXPERIMENTS_PATH = PROJECT_ROOT / "models" / "validation_experiments.json"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Datensatz fehlt: {DATA_PATH}")

df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
print(f"Daten: {len(df):,} Zeilen × {len(df.columns)} Spalten")
display(df.head(3))

Daten: 192,119 Zeilen × 27 Spalten


,date,open,high,low,close,volume,ticker,asset_type,source_file,daily_return,...,ma_200_distance,volatility_20d,rolling_high,drawdown,future_return_20d,target_20d,open_interest,volatility_60d,rolling_low_60,rolling_high_60
0,1985-06-21,0.25742,0.26381,0.25742,0.25742,46333854,AAPL,Stock,Stocks/aapl.us.txt,0.020374,...,-0.331288,0.040521,0.48791,-0.472403,0.044635,1.0,NaN,NaN,NaN,NaN
1,1985-06-24,0.27535,0.27920,0.27535,0.27535,57384755,AAPL,Stock,Stocks/aapl.us.txt,0.069653,...,-0.283328,0.040440,0.48791,-0.435654,-0.041910,0.0,NaN,NaN,NaN,NaN
2,1985-06-25,0.27920,0.28556,0.27920,0.27920,81966629,AAPL,Stock,Stocks/aapl.us.txt,0.013982,...,-0.271961,0.037120,0.48791,-0.427763,-0.073424,0.0,NaN,NaN,NaN,NaN


In [2]:
components = pd.DataFrame([
    ["Marktanalyse", "Kurs, Moving Averages, Candlesticks, Volatilität, Drawdown"],
    ["ML Insights", "Vergleich, Konfusionsmatrix, ROC/PR, Lernkurve, Wichtigkeiten"],
    ["Kapital-Kompass", "Positionsgröße auf Basis eines expliziten Risikobudgets"],
    ["Portfolio-Simulator", "Szenarien statt Renditeversprechen"],
    ["Lernstudio", "Quiz, Feedback und arsnova.eu-kompatibler Export"],
    ["Methodik & Export", "QUA³CK, Model Card und reproduzierbare Dateien"],
    ["Assistent", "Erklärt Analysekontext; erteilt keine Anlageberatung"],
], columns=["Bereich", "Wissenstransfer"])
components

,Bereich,Wissenstransfer
0,Marktanalyse,"Kurs, Moving Averages, Candlesticks, Volatilit..."
1,ML Insights,"Vergleich, Konfusionsmatrix, ROC/PR, Lernkurve..."
2,Kapital-Kompass,Positionsgröße auf Basis eines expliziten Risi...
3,Portfolio-Simulator,Szenarien statt Renditeversprechen
4,Lernstudio,"Quiz, Feedback und arsnova.eu-kompatibler Export"
5,Methodik & Export,"QUA³CK, Model Card und reproduzierbare Dateien"
6,Assistent,Erklärt Analysekontext; erteilt keine Anlagebe...


In [3]:
required_files = [
    "app.py",
    "src/pages/market.py",
    "src/pages/ml_insights.py",
    "src/pages/kompass.py",
    "src/pages/simulator.py",
    "src/pages/learning_studio.py",
    "src/pages/methodology.py",
    "docs/model_card.md",
]
pd.DataFrame({
    "Artefakt": required_files,
    "vorhanden": [(PROJECT_ROOT / path).exists() for path in required_files],
})

,Artefakt,vorhanden
0,app.py,True
1,src/pages/market.py,True
2,src/pages/ml_insights.py,True
3,src/pages/kompass.py,True
4,src/pages/simulator.py,True
5,src/pages/learning_studio.py,True
6,src/pages/methodology.py,True
7,docs/model_card.md,True


In [4]:
print("Lokaler Start:")
print("python -m streamlit run app.py")
print("\nDidaktische Referenzen:")
print("https://arsnova.eu/de/")
print("https://mc-test.streamlit.app/")

Lokaler Start:
python -m streamlit run app.py

Didaktische Referenzen:
https://arsnova.eu/de/
https://mc-test.streamlit.app/


## Transferprinzip

„KI damals und heute“ wird nicht nur erzählt: Der identische Test zeigt
historische Modellgenerationen, während der moderne Assistent ausschließlich
erklärt. Damit bleibt nachvollziehbar, welche Komponente rechnet, welche
visualisiert und welche Sprache erzeugt.

## Management-Checkpoint

Wissenstransfer ist bei einem Negativergebnis keine Kür, sondern die eigentliche
Prüfung: Die App muss ein schwaches Modell erklären, ohne es zu beschönigen und
ohne es zu verstecken. Der belastbare Nachweis dafür ist technisch, nicht
rhetorisch — **kein einziger Kennwert ist in `src/` hartkodiert**. Was die App
anzeigt, steht so in den Artefakten, aus denen auch Ausarbeitung, Poster und
diese Notebooks lesen.